# 03 — Diffusion Model Training

Load the clustered data, build the `DiffusionTransformer1D`, train with `Trainer`, and inspect generated samples mid-run.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import equinox as eqx

from src.data.loader    import load_raw, compute_stats, normalize, denormalize
from src.data.dataset   import make_windows, train_val_split, numpy_dataloader
from src.models.transformer1d import DiffusionTransformer1D
from src.models.diffusion     import DiffusionProcess
from src.training.train       import Trainer

plt.rcParams['figure.dpi'] = 110
print('JAX devices:', jax.devices())

# ── Training budget ──────────────────────────────────────────────────────────
# CPU-only: ~0.13 s/step, ~4 675 steps/epoch → ~10 min/epoch.
# Set QUICK_RUN=True to smoke-test 3 epochs (~30 min); False for real training.
# For full convergence (~200 epochs) run the provided train_full.py script with GPU.
QUICK_RUN  = True
N_EPOCHS   = 5    if QUICK_RUN else 200
BATCH_SIZE = 64
LR         = 1e-3
WARMUP_STEPS   = 300   if QUICK_RUN else 2000
GUIDANCE_SCALE = 1.5

CHECKPOINT_NAME = 'best_model.pkl'  # loaded if it exists; overwritten when done


ERROR:2026-03-26 15:28:01,269:jax._src.xla_bridge:444: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/nicola/Desktop/Supsi/tesina/.venv/lib/python3.10/site-packages/jax/_src/xla_bridge.py", line 442, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/nicola/Desktop/Supsi/tesina/.venv/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 324, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/nicola/Desktop/Supsi/tesina/.venv/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 281, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CUDA libraries.


JAX devices: [CpuDevice(id=0)]


## 1. Load & prepare data

In [2]:

df = load_raw('../data/power.pk')

clusters_df = pd.read_csv('../data/clusters.csv')
cluster_labels = clusters_df['cluster_id'].values            # (N,)
N_CLUSTERS = int(cluster_labels.max()) + 1

print(f'{df.shape[1]} meters, {N_CLUSTERS} clusters')
print(f'Cluster sizes: { {c: (cluster_labels==c).sum() for c in range(N_CLUSTERS)} }')

# Use the DatetimeIndex already present in power.pk
timestamps = df.index if isinstance(df.index, pd.DatetimeIndex) else None

stats = compute_stats(df, cluster_labels)
df_norm = normalize(df, stats, cluster_labels)


321 meters, 3 clusters
Cluster sizes: {0: np.int64(183), 1: np.int64(34), 2: np.int64(104)}


In [3]:
xs, cs, mid = make_windows(df_norm, cluster_labels, timestamps)
print(f'Windows: xs={xs.shape}  cs={cs.shape}  mid={mid.shape}')

x_train, c_train, x_val, c_val = train_val_split(xs, cs, mid, n_meters=df.shape[1])
print(f'Train: {x_train.shape[0]}  Val: {x_val.shape[0]}')

# ── Per-cluster window count ──
print('\nWindows per cluster (train):')
for cid in range(N_CLUSTERS):
    n_tr = (c_train[:, 0] == cid).sum()
    n_va = (c_val[:,   0] == cid).sum()
    wd   = (c_train[:, 1] == 0).sum()
    we   = (c_train[:, 1] == 1).sum()
    print(f'  Cluster {cid}: train={n_tr:6d}  val={n_va:5d}  (all: weekday={wd}, weekend={we})')

# ── Sanity: check z-score normalisation per cluster ──
print('\nNormalised value stats per cluster (train, should be ~mean=0, std=1):')
for cid in range(N_CLUSTERS):
    mask = c_train[:, 0] == cid
    vals = x_train[mask].ravel()
    print(f'  Cluster {cid}: mean={vals.mean():+.3f}  std={vals.std():.3f}  '
          f'min={vals.min():.3f}  max={vals.max():.3f}')


Windows: xs=(351816, 24)  cs=(351816, 2)  mid=(351816,)
Train: 299208  Val: 52608

Windows per cluster (train):
  Cluster 0: train=169880  val=30688  (all: weekday=213759, weekend=85449)
  Cluster 1: train= 31784  val= 5480  (all: weekday=213759, weekend=85449)
  Cluster 2: train= 97544  val=16440  (all: weekday=213759, weekend=85449)

Normalised value stats per cluster (train, should be ~mean=0, std=1):
  Cluster 0: mean=+0.025  std=1.077  min=-0.306  max=26.476
  Cluster 1: mean=+0.031  std=1.080  min=-0.212  max=17.986
  Cluster 2: mean=+0.024  std=1.066  min=-0.449  max=11.326


## 2. Model & diffusion setup

In [4]:

key = jax.random.PRNGKey(0)

model = DiffusionTransformer1D(
    seq_len    = 24,
    d_model    = 128,
    n_heads    = 4,
    n_layers   = 4,
    d_ff       = 256,
    n_clusters = N_CLUSTERS,
    n_day_types= 2,
    key        = key,
)

diffusion = DiffusionProcess(T=1000, freq_loss_weight=0.05)

total_steps = N_EPOCHS * max(1, len(x_train) // BATCH_SIZE)
trainer = Trainer(model, diffusion, lr=LR, warmup_steps=WARMUP_STEPS,
                  total_steps=total_steps, checkpoint_dir='../checkpoints')

n_params = sum(x.size for x in jax.tree_util.tree_leaves(eqx.filter(model, eqx.is_array)))
print(f'Model parameters : {n_params:,}')
steps_per_epoch = max(1, len(x_train) // BATCH_SIZE)
print(f'Steps per epoch  : {steps_per_epoch}')
print(f'Total steps      : {total_steps}')
print(f'Epochs           : {N_EPOCHS}  (QUICK_RUN={QUICK_RUN})')

# ── Load existing checkpoint if available ──
import os
ckpt_path = f'../checkpoints/{CHECKPOINT_NAME}'
if os.path.exists(ckpt_path):
    trainer.load(CHECKPOINT_NAME)
    print(f'\nResumed from checkpoint at step {trainer.step}.')
else:
    print('\nNo checkpoint found — will train from scratch.')


Model parameters : 845,890
Steps per epoch  : 4675
Total steps      : 23375
Epochs           : 5  (QUICK_RUN=True)

No checkpoint found — will train from scratch.


## 3. Training

In [5]:
train_loader = numpy_dataloader(x_train, c_train, batch_size=BATCH_SIZE, shuffle=True,  rng=np.random.default_rng(1))
val_loader   = numpy_dataloader(x_val,   c_val,   batch_size=BATCH_SIZE, shuffle=False, rng=np.random.default_rng(2))

trainer.fit(
    train_loader    = train_loader,
    val_loader      = val_loader,
    n_epochs        = N_EPOCHS,
    val_every       = 1,
    save_every      = max(1, N_EPOCHS // 5),
    log_every_steps = max(1, len(x_train) // BATCH_SIZE // 4),
    val_batches     = max(1, len(x_val) // BATCH_SIZE),
)
train_losses = trainer.train_losses
val_losses   = trainer.val_losses
print(f'\nTraining complete. Final train loss: {train_losses[-1]:.4f}  val loss: {val_losses[-1]:.4f}')


  step   1168  loss 0.9491
  step   2336  loss 0.9278
  step   3504  loss 1.2059


KeyboardInterrupt: 

## 4. Loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs = range(1, len(train_losses) + 1)
axes[0].plot(list(epochs), train_losses, 'o-', color='steelblue', linewidth=1.5, label='Train')
axes[0].plot(list(epochs), val_losses,   's-', color='coral',     linewidth=1.5, label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Noise-prediction loss')
axes[0].set_title('Per-epoch training and validation loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Relative improvement curve (useful for detecting convergence)
if len(train_losses) > 1:
    rel_improvement = [(train_losses[0] - l) / train_losses[0] * 100 for l in train_losses]
    axes[1].plot(list(epochs), rel_improvement, 'o-', color='mediumseagreen', linewidth=1.5)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('% reduction from epoch-1 loss')
    axes[1].set_title('Relative loss improvement')
    axes[1].grid(True, alpha=0.3)
    axes[1].axhline(0, color='grey', linestyle='--', linewidth=0.8)

plt.tight_layout(); plt.show()
print(f'Epoch 1 loss: {train_losses[0]:.4f}')
print(f'Final  loss : {train_losses[-1]:.4f}  ({(train_losses[0]-train_losses[-1])/train_losses[0]*100:.1f}% reduction)')
if QUICK_RUN:
    print('\n⚠ QUICK_RUN=True: only 5 epochs trained. For thesis-quality results,')
    print('  set QUICK_RUN=False and run on GPU (~200 epochs, several hours).')


## 5. Quick sample inspection (DDIM 50 steps)

In [ ]:

hours  = np.arange(24)
n_show = 4

fig, axes = plt.subplots(N_CLUSTERS, 4, figsize=(16, 3.5 * N_CLUSTERS))
day_labels = ['Weekday', 'Weekend']

for cid in range(N_CLUSTERS):
    for dt in range(2):
        c_batch  = jnp.array([[cid, dt]] * n_show, dtype=jnp.int32)
        gen_key  = jax.random.PRNGKey(cid * 10 + dt)
        samples_norm = diffusion.ddim_sample(
            trainer.model, c_batch,
            seq_len=24, batch_size=n_show,
            key=gen_key, n_steps=50, guidance_scale=GUIDANCE_SCALE
        )  # (n_show, 24) — normalised
        samples_norm = np.array(samples_norm)

        # Denormalise to physical units
        samples_phys = denormalize(samples_norm, cluster_id=cid, stats=stats)

        ax_norm = axes[cid, dt]
        ax_phys = axes[cid, dt + 2]

        for i in range(n_show):
            ax_norm.plot(hours, samples_norm[i], alpha=0.7, linewidth=1)
            ax_phys.plot(hours, samples_phys[i], alpha=0.7, linewidth=1)

        ax_norm.set_title(f'C{cid} {day_labels[dt]} (z-scored)', fontsize=9)
        ax_norm.set_xlabel('Hour'); ax_norm.set_xticks(range(0, 24, 6))

        ax_phys.set_title(f'C{cid} {day_labels[dt]} (Wh)', fontsize=9)
        ax_phys.set_xlabel('Hour'); ax_phys.set_xticks(range(0, 24, 6))
        ax_phys.set_ylim(bottom=0)

        for ax in [ax_norm, ax_phys]:
            if cid == 0 and dt in [0, 2]:
                pass

# Add real data overlay for comparison
for cid in range(N_CLUSTERS):
    mask     = (c_train[:, 0] == cid) & (c_train[:, 1] == 0)  # weekday
    real_sub = x_train[mask][:n_show]
    for i in range(min(n_show, len(real_sub))):
        axes[cid, 0].plot(hours, real_sub[i], color='grey', alpha=0.4,
                          linewidth=0.8, linestyle='--')

# Add legend to first panel
from matplotlib.lines import Line2D
axes[0, 0].legend(
    handles=[Line2D([0], [0], color='grey', linestyle='--', label='Real (z-scored)')],
    fontsize=7, loc='upper left')

plt.suptitle('DDIM samples (left 2 cols: z-scored | right 2 cols: physical units)\n'
             '— grey dashed = real training samples for reference',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.show()


## 6. Save final checkpoint

In [ ]:

trainer.save(CHECKPOINT_NAME)

n_params = sum(x.size for x in jax.tree_util.tree_leaves(eqx.filter(trainer.model, eqx.is_array)))
print(f"  Parameters : {n_params:,}")
print(f"  Epochs done: {N_EPOCHS}  (QUICK_RUN={QUICK_RUN})")
if trainer.val_losses:
    print(f"  Best val   : {min(trainer.val_losses):.4f}  (epoch {np.argmin(trainer.val_losses)+1})")


## 7. Observations


In [ ]:

# ── §7 Training observations ─────────────────────────────────────────────────

print("=" * 60)
print("TRAINING RUN SUMMARY")
print("=" * 60)

print(f"\n  Epochs        : {N_EPOCHS}  (QUICK_RUN={QUICK_RUN})")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Learning rate : {LR}  (cosine schedule, warmup={WARMUP_STEPS})")
print(f"  Steps/epoch   : {max(1, len(x_train) // BATCH_SIZE)}")
print(f"  Model params  : {sum(x.size for x in jax.tree_util.tree_leaves(eqx.filter(trainer.model, eqx.is_array))):,}")

if trainer.train_losses:
    first, last = trainer.train_losses[0], trainer.train_losses[-1]
    pct = (first - last) / first * 100
    print(f"\n  Initial loss  : {first:.4f}")
    print(f"  Final loss    : {last:.4f}  ({pct:.1f}% reduction)")
    best_epoch = int(np.argmin(trainer.val_losses)) + 1 if trainer.val_losses else "—"
    best_val   = min(trainer.val_losses)               if trainer.val_losses else float('nan')
    print(f"  Best val loss : {best_val:.4f}  (epoch {best_epoch})")

print(f"\n  Training device: {jax.devices()[0]}")
print(f"\n  Next step: run 04_evaluation.ipynb after full training on GPU (~200 epochs).")
print("  Expected full-training metric targets:")
print("    Discriminative accuracy ≤ 0.55  (good sample quality)")
print("    ACF L2 distance         ≤ 0.05  (temporal structure preserved)")
